# spharmgrid — ERA5 850-hPa spherical harmonic explorer

This self-contained example compares the original ERA5 850-hPa wind with a processed wind field on the same timestamp. The processed field is a vector spherical harmonic selection or taper, optionally followed by regridding to a fixed Gauss–Legendre grid.

The same notebook runs interactively in JupyterLab and as a local Panel application. spharmgrid supplies the numerical spherical harmonic operations through `ducc0`; this example supplies only the data cache, scientific display, and application layer. See the [filtering](../../docs/filtering.md), [regridding](../../docs/regridding.md), [kinematics](../../docs/kinematics.md), and [operators](../../docs/operators.md) documentation for the underlying conventions.

## Imports and the immutable data-release cache

In [ ]:
from functools import lru_cache
from pathlib import Path

import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import numpy as np
import panel as pn
import pooch
import xarray as xr

import spharmgrid as sg

pn.extension()
hv.extension("bokeh")

DATA_RELEASE = "v0.2.0-data"
DATA_BASE_URL = (
    f"https://github.com/mwyau/PyStormTracker-Data/releases/download/{DATA_RELEASE}/"
)
DATA_CACHE = Path(pooch.os_cache("spharmgrid")) / "interactive-v0.2.0-data"
UV_FILENAME = "era5_uv850_2025-2026_djf_2.5x2.5.nc"
VO_FILENAME = "era5_vo850_2025-2026_djf_2.5x2.5.nc"
DATA_REGISTRY = {
    UV_FILENAME: (
        "sha256:43cbc346a52c5230ac34eb22c7a640800fbffad40da4058686c8042a76bc5965"
    ),
    VO_FILENAME: (
        "sha256:46ce78cd3b065d3777c2d628cdc2311d68a9fcb4d3a3b9948db7c7376ae7a6aa"
    ),
}
DATA = pooch.create(
    path=DATA_CACHE,
    base_url=DATA_BASE_URL,
    registry=DATA_REGISTRY,
)


def fetch_asset(filename: str) -> Path:
    """Fetch one pinned release asset into Pooch's user cache."""
    try:
        return Path(DATA.fetch(filename, progressbar=False))
    except Exception as exc:
        # Pooch uses downloader-specific exception types; keep the error at
        # this asset boundary actionable without hiding computation errors.
        expected_hash = DATA_REGISTRY[filename].split(":", 1)[1]
        raise RuntimeError(
            f"Could not retrieve {filename} from immutable PyStormTracker-Data "
            f"{DATA_RELEASE}. Check network access; expected SHA-256 is "
            f"{expected_hash}."
        ) from exc

## Load ERA5 and inspect the actual schema

The pinned wind asset is inspected before it is passed to spharmgrid. The release contains one singleton pressure level, so only that genuinely singleton dimension is reduced. The resulting `u` and `v` arrays retain the actual `valid_time`, latitude, and longitude coordinates.

In [ ]:
uv_path = fetch_asset(UV_FILENAME)
with xr.open_dataset(uv_path, engine="h5netcdf") as raw:
    missing = {"u", "v"} - set(raw.data_vars)
    if missing:
        raise ValueError(
            f"ERA5 wind asset is missing required variable(s): {sorted(missing)}"
        )
    expected_dims = ("valid_time", "pressure_level", "latitude", "longitude")
    for name in ("u", "v"):
        if raw[name].dims != expected_dims:
            raise ValueError(
                f"Unexpected ERA5 {name} dimensions {raw[name].dims!r}; "
                f"expected {expected_dims!r}."
            )
    if raw.sizes["pressure_level"] != 1:
        raise ValueError("The pinned ERA5 example expects one pressure level.")
    pressure = float(raw["pressure_level"].values[0])
    if not np.isclose(pressure, 850.0):
        raise ValueError(f"Expected an 850-hPa asset, found {pressure:g} hPa.")
    raw_schema = {
        "data_vars": list(raw.data_vars),
        "dimensions": {str(name): int(size) for name, size in raw.sizes.items()},
        "variables": {
            name: {
                "dims": tuple(var.dims),
                "shape": tuple(int(size) for size in var.shape),
                "units": var.attrs.get("units"),
                "standard_name": var.attrs.get("standard_name"),
            }
            for name, var in raw.data_vars.items()
        },
        "coordinates": {
            name: {
                "dims": tuple(coord.dims),
                "shape": tuple(int(size) for size in coord.shape),
                "first": str(coord.values.flat[0]) if coord.size else None,
                "last": str(coord.values.flat[-1]) if coord.size else None,
                "standard_name": coord.attrs.get("standard_name"),
                "units": coord.attrs.get("units"),
            }
            for name, coord in raw.coords.items()
        },
        "global_attributes": {
            name: raw.attrs.get(name)
            for name in ("Conventions", "institution", "history")
            if name in raw.attrs
        },
    }
    ERA5 = raw[["u", "v"]].isel(pressure_level=0, drop=True).load()

for name in ("u", "v"):
    if ERA5[name].dims != ("valid_time", "latitude", "longitude"):
        raise ValueError(
            f"After reducing pressure_level, {name} is not a time series of "
            "2-D latitude-longitude fields."
        )

TIMES = ERA5["valid_time"].values
schema_summary = {
    "cached_file": str(uv_path),
    "file_size_mb": round(uv_path.stat().st_size / 1e6, 2),
    "release": DATA_RELEASE,
    "schema": raw_schema,
    "loaded_field_dims": {name: ERA5[name].dims for name in ("u", "v")},
}
schema_summary

## Detect the source grid and derive the usable spectral range

`sg.detect_grid()` is applied to an actual ERA5 field. The triangular limit below is a small notebook-local calculation from the documented GL/CC dimension limits; it does not use a private spharmgrid helper.

In [ ]:
def triangular_limit(grid: sg.Grid) -> int:
    latitude_lmax = grid.nlat - 2 if grid.kind == "cc" else grid.nlat - 1
    longitude_mmax = (grid.nlon - 1) // 2
    return min(latitude_lmax, longitude_mmax)


source_grid = sg.detect_grid(ERA5["u"].isel(valid_time=0))
if source_grid.kind != "cc":
    raise ValueError(
        f"Expected the pinned pole-including ERA5 grid to be CC; "
        f"sg.detect_grid() identified {source_grid.kind!r}."
    )
if ERA5["u"].isel(valid_time=0).ndim != 2 or ERA5["v"].isel(valid_time=0).ndim != 2:
    raise ValueError("Each ERA5 timestamp must be a 2-D global wind field.")

source_lmax = triangular_limit(source_grid)
if source_lmax < 42:
    raise ValueError(
        f"The detected ERA5 grid supports only T{source_lmax}; T42 is required "
        "for the default demonstration."
    )
source_latitude_order = (
    "descending" if source_grid.latitude[0] > source_grid.latitude[-1] else "ascending"
)
gl_target = sg.gaussian_grid(
    source_grid.nlat - 1,
    source_grid.nlon,
    lon0=float(source_grid.longitude[0]),
    latitude_order=source_latitude_order,
)
gl_lmax = triangular_limit(gl_target)
if gl_lmax < source_lmax:
    raise ValueError(
        f"The fixed GL target supports only T{gl_lmax}, below the source "
        f"grid limit T{source_lmax}."
    )
DEFAULT_LMAX = min(42, source_lmax)

grid_summary = {
    "detected_source_grid": source_grid.kind.upper(),
    "source_shape": (source_grid.nlat, source_grid.nlon),
    "source_latitude_order": source_latitude_order,
    "source_longitude_range": (
        float(source_grid.longitude.min()),
        float(source_grid.longitude.max()),
    ),
    "source_triangular_limit": f"T{source_lmax}",
    "fixed_gl_target_shape": (gl_target.nlat, gl_target.nlon),
    "fixed_gl_triangular_limit": f"T{gl_lmax}",
    "default_range": f"T0-{DEFAULT_LMAX}",
}
grid_summary

## Accessor ergonomics

The reactive application uses direct public functions so that its data flow is explicit. The same ERA5 Dataset also supports the xarray accessor style. This cell computes one small, early demonstration rather than maintaining separate accessor and direct implementations.

In [ ]:
accessor_frame = ERA5.isel(valid_time=0)
accessor_kinematics = accessor_frame.sg.kinematics()
accessor_potentials = accessor_frame.sg.potentials()
{
    "accessor_grid_type": accessor_frame.sg.grid_type,
    "kinematics_variables": list(accessor_kinematics.data_vars),
    "potential_variables": list(accessor_potentials.data_vars),
    "timestamp": str(TIMES[0]),
}

## Processing model

The right-hand wind is always processed as a vector field with `sg.regrid_vector(u, v, ...)`. Selecting the ERA5 CC grid as the target intentionally gives a vector spherical harmonic analysis, degree selection or taper, and inverse vector synthesis on the same physical grid. The geographic components are never filtered as unrelated scalar fields.

With tapering off, `T0–42` is a hard selection that retains total degrees 0 through 42; `T6–42` removes degrees 0 through 5. With tapering on, `taper=0.1` means the retained endpoint has response 0.1 under the documented Sardeshmukh–Hoskins taper; it is not a cutoff. The fixed GL target is created with the public `sg.gaussian_grid()` constructor and checked against the selected spectral range.

In [ ]:
DIAGNOSTICS = (
    "Wind",
    "Relative vorticity",
    "Divergence",
    "Streamfunction",
    "Velocity potential",
    "Rotational wind",
    "Divergent wind",
)
GRID_MODES = ("ERA5 CC", "Gauss–Legendre")


@lru_cache(maxsize=72)
def get_frame(time_index: int) -> tuple[xr.DataArray, xr.DataArray]:
    index = int(time_index)
    if not 0 <= index < len(TIMES):
        raise IndexError(f"Time index {index} is outside 0..{len(TIMES) - 1}.")
    frame = ERA5.isel(valid_time=index)
    return frame["u"], frame["v"]


@lru_cache(maxsize=36)
def process_wind(
    time_index: int,
    lmin: int,
    lmax: int,
    taper: float | None,
    output_grid: str,
) -> xr.Dataset:
    if not 0 <= lmin <= lmax <= source_lmax:
        raise ValueError(
            f"Spectral range T{lmin}-{lmax} is outside the detected source "
            f"limit T{source_lmax}."
        )
    if output_grid == "ERA5 CC":
        target = source_grid
    elif output_grid == "Gauss–Legendre":
        target = gl_target
    else:
        raise ValueError(f"Unknown output grid {output_grid!r}.")
    target_lmax = triangular_limit(target)
    if lmax > target_lmax:
        raise ValueError(
            f"Spectral range T{lmin}-{lmax} is not representable on the "
            f"{output_grid} target (limit T{target_lmax})."
        )
    u_frame, v_frame = get_frame(int(time_index))
    return sg.regrid_vector(
        u_frame,
        v_frame,
        target,
        lmin=lmin,
        lmax=lmax,
        taper=taper,
    )


def wind_state(u: xr.DataArray, v: xr.DataArray, label: str) -> dict[str, object]:
    speed = np.hypot(u, v)
    speed.name = "wind_speed"
    speed.attrs = {"long_name": f"{label} speed", "units": "m s-1"}
    return {
        "plot_kind": "wind",
        "field": speed,
        "u": u,
        "v": v,
        "display_scale": 1.0,
        "display_label": f"{label} speed (m s⁻¹)",
    }


def scalar_state(
    field: xr.DataArray, scale: float, display_label: str
) -> dict[str, object]:
    return {
        "plot_kind": "scalar",
        "field": field,
        "display_scale": scale,
        "display_label": display_label,
    }


def compute_diagnostic(
    u: xr.DataArray, v: xr.DataArray, diagnostic: str
) -> dict[str, object]:
    if diagnostic == "Wind":
        return wind_state(u, v, "Wind")
    if diagnostic in {"Relative vorticity", "Divergence"}:
        kin = sg.kinematics(u, v)
        if diagnostic == "Relative vorticity":
            return scalar_state(kin["vo"], 1e5, "Relative vorticity (10⁻⁵ s⁻¹)")
        return scalar_state(kin["d"], 1e5, "Divergence (10⁻⁵ s⁻¹)")
    if diagnostic in {"Streamfunction", "Velocity potential"}:
        potentials = sg.potentials(u, v)
        if diagnostic == "Streamfunction":
            return scalar_state(potentials["strf"], 1e-6, "Streamfunction (10⁶ m² s⁻¹)")
        return scalar_state(potentials["vp"], 1e-6, "Velocity potential (10⁶ m² s⁻¹)")
    kin = sg.kinematics(u, v)
    if diagnostic == "Rotational wind":
        components = sg.rotational_wind(kin["vo"], quantity="vorticity")
        return wind_state(
            components["u_rotational"],
            components["v_rotational"],
            "Rotational wind",
        )
    if diagnostic == "Divergent wind":
        components = sg.divergent_wind(kin["d"], quantity="divergence")
        return wind_state(
            components["u_divergent"],
            components["v_divergent"],
            "Divergent wind",
        )
    raise ValueError(f"Unknown diagnostic {diagnostic!r}.")


@lru_cache(maxsize=72)
def original_diagnostic(time_index: int, diagnostic: str) -> dict[str, object]:
    return compute_diagnostic(*get_frame(int(time_index)), diagnostic)


@lru_cache(maxsize=72)
def processed_diagnostic(
    time_index: int,
    lmin: int,
    lmax: int,
    taper: float | None,
    output_grid: str,
    diagnostic: str,
) -> dict[str, object]:
    wind = process_wind(time_index, lmin, lmax, taper, output_grid)
    return compute_diagnostic(wind["u"], wind["v"], diagnostic)

## On-demand consistency checks

The checks below run only when the button is clicked. They use the current processed field for the selected timestamp, spectral range, taper, and output grid. Relative RMS is `RMS(a - b) / RMS(a)` with an explicit zero-norm convention. The optional ERA5 `vo` asset is fetched only inside the external-reference check and is compared against the unprocessed ERA5 CC frame.

In [ ]:
# ruff: noqa: E501
def field_values(field: xr.DataArray) -> np.ndarray:
    return np.asarray(field.values, dtype=np.float64)


def relative_rms(reference: xr.DataArray, candidate: xr.DataArray) -> float:
    a = field_values(reference).ravel()
    b = field_values(candidate).ravel()
    valid = np.isfinite(a) & np.isfinite(b)
    if not np.any(valid):
        return float("nan")
    numerator = float(np.sqrt(np.mean((a[valid] - b[valid]) ** 2)))
    denominator = float(np.sqrt(np.mean(a[valid] ** 2)))
    if denominator == 0.0:
        return 0.0 if numerator == 0.0 else float("inf")
    return numerator / denominator


def vector_relative_rms(
    reference_u: xr.DataArray,
    reference_v: xr.DataArray,
    candidate_u: xr.DataArray,
    candidate_v: xr.DataArray,
) -> float:
    a_u = field_values(reference_u).ravel()
    a_v = field_values(reference_v).ravel()
    b_u = field_values(candidate_u).ravel()
    b_v = field_values(candidate_v).ravel()
    valid = np.isfinite(a_u) & np.isfinite(a_v) & np.isfinite(b_u) & np.isfinite(b_v)
    if not np.any(valid):
        return float("nan")
    numerator = float(
        np.sqrt(
            np.mean((a_u[valid] - b_u[valid]) ** 2 + (a_v[valid] - b_v[valid]) ** 2)
        )
    )
    denominator = float(np.sqrt(np.mean(a_u[valid] ** 2 + a_v[valid] ** 2)))
    if denominator == 0.0:
        return 0.0 if numerator == 0.0 else float("inf")
    return numerator / denominator


def correlation(left: xr.DataArray, right: xr.DataArray) -> float:
    a = field_values(left).ravel()
    b = field_values(right).ravel()
    valid = np.isfinite(a) & np.isfinite(b)
    if (
        np.count_nonzero(valid) < 2
        or np.std(a[valid]) == 0.0
        or np.std(b[valid]) == 0.0
    ):
        return float("nan")
    return float(np.corrcoef(a[valid], b[valid])[0, 1])


def format_metric(value: float) -> str:
    return "n/a" if not np.isfinite(value) else f"{value:.3e}"


def timestamp_label(time_index: int) -> str:
    timestamp = np.datetime_as_string(TIMES[int(time_index)], unit="m")
    return f"{timestamp.replace('T', ' ')} UTC"


_ERA5_VO: xr.DataArray | None = None


def load_era5_vo() -> xr.DataArray:
    global _ERA5_VO
    if _ERA5_VO is not None:
        return _ERA5_VO
    vo_path = fetch_asset(VO_FILENAME)
    with xr.open_dataset(vo_path, engine="h5netcdf") as raw:
        if "vo" not in raw.data_vars:
            raise ValueError("ERA5 vorticity asset does not contain a vo variable.")
        expected_dims = ("valid_time", "pressure_level", "latitude", "longitude")
        if raw["vo"].dims != expected_dims or raw.sizes.get("pressure_level") != 1:
            raise ValueError(
                f"Unexpected ERA5 vo schema: dims={raw['vo'].dims!r}, "
                f"sizes={dict(raw.sizes)!r}."
            )
        pressure = float(raw["pressure_level"].values[0])
        if not np.isclose(pressure, 850.0):
            raise ValueError(f"Expected vo at 850 hPa, found {pressure:g} hPa.")
        reference = raw["vo"].isel(pressure_level=0, drop=True).load()
    if not np.array_equal(reference["valid_time"].values, ERA5["valid_time"].values):
        raise ValueError("ERA5 vo timestamps do not match the primary u/v asset.")
    for axis in ("latitude", "longitude"):
        if not np.array_equal(reference[axis].values, ERA5[axis].values):
            raise ValueError(f"ERA5 vo {axis} coordinates do not match the u/v asset.")
    if sg.detect_grid(reference.isel(valid_time=0)).kind != source_grid.kind:
        raise ValueError(
            "ERA5 vo is not on the same detected CC grid as the u/v asset."
        )
    _ERA5_VO = reference
    return _ERA5_VO


def external_vo_metrics(time_index: int) -> dict[str, float]:
    reference = load_era5_vo().isel(valid_time=int(time_index))
    calculated = sg.vorticity(*get_frame(int(time_index)))
    difference = calculated - reference
    return {
        "correlation": correlation(calculated, reference),
        "rms_difference": float(np.sqrt(np.nanmean(field_values(difference) ** 2))),
        "mean_difference": float(np.nanmean(field_values(difference))),
        "reference_rms": float(np.sqrt(np.nanmean(field_values(reference) ** 2))),
    }


def run_consistency_checks(
    time_index: int,
    lmin: int,
    lmax: int,
    taper: float | None,
    output_grid: str,
) -> str:
    processed = process_wind(time_index, lmin, lmax, taper, output_grid)
    u, v = processed["u"], processed["v"]
    kin = sg.kinematics(u, v)
    pot = sg.potentials(u, v)
    parts = sg.helmholtz(u, v)
    restored_wind = sg.wind(kin["vo"], kin["d"], source="vorticity_divergence")
    lap_strf = sg.laplacian(pot["strf"])
    lap_vp = sg.laplacian(pot["vp"])
    inv_strf = sg.inverse_laplacian(kin["vo"])
    inv_vp = sg.inverse_laplacian(kin["d"])
    gradient_vp = sg.gradient(pot["vp"])
    inverse_gradient_vp = sg.inverse_gradient(
        gradient_vp["gradient_eastward"], gradient_vp["gradient_northward"]
    )
    divergent_from_vp = sg.divergent_wind(pot["vp"], quantity="velocity_potential")
    rotational_from_vo = sg.rotational_wind(kin["vo"], quantity="vorticity")
    divergent_from_d = sg.divergent_wind(kin["d"], quantity="divergence")
    vector_lap = sg.vector_laplacian(u, v)
    inverse_vector_lap = sg.inverse_vector_laplacian(vector_lap["u"], vector_lap["v"])

    metrics = [
        (
            "wind(vo, d) reconstruction",
            vector_relative_rms(u, v, restored_wind["u"], restored_wind["v"]),
        ),
        (
            "Helmholtz component sum",
            vector_relative_rms(
                u,
                v,
                parts["u_divergent"] + parts["u_rotational"],
                parts["v_divergent"] + parts["v_rotational"],
            ),
        ),
        ("laplacian(strf) versus vo", relative_rms(kin["vo"], lap_strf)),
        ("laplacian(vp) versus d", relative_rms(kin["d"], lap_vp)),
        ("inverse_laplacian(vo) versus strf", relative_rms(pot["strf"], inv_strf)),
        ("inverse_laplacian(d) versus vp", relative_rms(pot["vp"], inv_vp)),
        (
            "gradient(vp) versus divergent_wind(vp)",
            vector_relative_rms(
                gradient_vp["gradient_eastward"],
                gradient_vp["gradient_northward"],
                divergent_from_vp["u_divergent"],
                divergent_from_vp["v_divergent"],
            ),
        ),
        (
            "inverse_gradient(gradient(vp)) versus vp",
            relative_rms(pot["vp"], inverse_gradient_vp),
        ),
        (
            "rotational_wind(vo) versus Helmholtz rotational",
            vector_relative_rms(
                parts["u_rotational"],
                parts["v_rotational"],
                rotational_from_vo["u_rotational"],
                rotational_from_vo["v_rotational"],
            ),
        ),
        (
            "divergent_wind(d) versus Helmholtz divergent",
            vector_relative_rms(
                parts["u_divergent"],
                parts["v_divergent"],
                divergent_from_d["u_divergent"],
                divergent_from_d["v_divergent"],
            ),
        ),
        (
            "rotational_wind(vo) + divergent_wind(d)",
            vector_relative_rms(
                u,
                v,
                rotational_from_vo["u_rotational"] + divergent_from_d["u_divergent"],
                rotational_from_vo["v_rotational"] + divergent_from_d["v_divergent"],
            ),
        ),
        (
            "inverse_vector_laplacian(vector_laplacian(wind))",
            vector_relative_rms(u, v, inverse_vector_lap["u"], inverse_vector_lap["v"]),
        ),
    ]

    taper_description = (
        "hard selection"
        if taper is None
        else f"Sardeshmukh–Hoskins endpoint = {taper:g}"
    )
    lines = [
        "### Consistency checks",
        f"Current processed field: **{timestamp_label(time_index)}**, `{output_grid}`, `T{lmin}–{lmax}`, {taper_description}.",
        "",
        "Relative RMS is `RMS(a - b) / RMS(a)`; no pass/fail threshold is applied.",
        "",
        "| Relationship | Relative RMS |",
        "|---|---:|",
    ]
    lines.extend(f"| {label} | {format_metric(value)} |" for label, value in metrics)
    lines.extend(
        [
            "",
            "The external comparison uses the matching unprocessed ERA5 CC frame; it is not a ground-truth or exact-parity test.",
        ]
    )
    try:
        external = external_vo_metrics(time_index)
    except (RuntimeError, ValueError) as exc:
        lines.append(f"\n**External ERA5 reference unavailable:** {exc}")
    else:
        lines.extend(
            [
                "",
                "| External ERA5 `vo` reference metric | Value |",
                "|---|---:|",
                f"| Correlation | {format_metric(external['correlation'])} |",
                f"| RMS difference (s⁻¹) | {format_metric(external['rms_difference'])} |",
                f"| Mean difference (s⁻¹) | {format_metric(external['mean_difference'])} |",
                f"| Reference RMS (s⁻¹) | {format_metric(external['reference_rms'])} |",
            ]
        )
    return chr(10).join(lines)

## Plot helpers

The scientific arrays remain on their original coordinates for every spharmgrid call. The `[-180, 180)` longitude normalization below is a plot-only copy used to make the seam less distracting. Scalar panels share one robust color limit computed from both the original and processed fields; vector arrows are downsampled only for drawing.

In [ ]:
MAP_CRS = ccrs.PlateCarree()
COASTLINE = gv.feature.coastline()
PLOT_STRIDE = 4


def plot_scalar_arrays(
    field: xr.DataArray, scale: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ordered = field.transpose("latitude", "longitude")
    longitude = (
        np.mod(np.asarray(ordered["longitude"].values, dtype=float) + 180.0, 360.0)
        - 180.0
    )
    order = np.argsort(longitude, kind="stable")
    values = np.asarray(ordered.values, dtype=float)[:, order] * scale
    return longitude[order], np.asarray(ordered["latitude"].values, dtype=float), values


def plot_wind_arrays(
    u: xr.DataArray, v: xr.DataArray
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    ordered_u = u.transpose("latitude", "longitude")
    ordered_v = v.transpose("latitude", "longitude")
    longitude = (
        np.mod(np.asarray(ordered_u["longitude"].values, dtype=float) + 180.0, 360.0)
        - 180.0
    )
    order = np.argsort(longitude, kind="stable")
    return (
        longitude[order],
        np.asarray(ordered_u["latitude"].values, dtype=float),
        np.asarray(ordered_u.values, dtype=float)[:, order],
        np.asarray(ordered_v.values, dtype=float)[:, order],
    )


def displayed_values(state: dict[str, object]) -> np.ndarray:
    field = state["field"]
    scale = float(state["display_scale"])
    return field_values(field) * scale


def common_color_limits(
    original: dict[str, object], processed: dict[str, object]
) -> tuple[float, float]:
    values = np.concatenate(
        (displayed_values(original).ravel(), displayed_values(processed).ravel())
    )
    values = values[np.isfinite(values)]
    if values.size == 0:
        return (-1.0, 1.0)
    if original["plot_kind"] == "scalar":
        limit = float(np.percentile(np.abs(values), 99.0))
        limit = max(limit, float(np.max(np.abs(values)))) if limit == 0.0 else limit
        return (-limit, limit)
    limit = float(np.percentile(values, 99.0))
    return (0.0, max(limit, float(np.max(values)), 1.0))


def map_options(title: str) -> dict[str, object]:
    return {
        "width": 620,
        "height": 360,
        "xlim": (-180, 180),
        "ylim": (-90, 90),
        "projection": MAP_CRS,
        "title": title,
    }


def make_wind_plot(
    state: dict[str, object], clim: tuple[float, float], title: str
) -> hv.Overlay:
    longitude, latitude, u_values, v_values = plot_wind_arrays(state["u"], state["v"])
    speed = np.hypot(u_values, v_values)
    mesh = gv.QuadMesh(
        (longitude, latitude, speed),
        kdims=["longitude", "latitude"],
        vdims=[state["display_label"]],
        crs=MAP_CRS,
    ).opts(cmap="Viridis", colorbar=True, clim=clim, **map_options(title))
    display_latitude = latitude[1:-1:PLOT_STRIDE]
    display_u = u_values[1:-1:PLOT_STRIDE, ::PLOT_STRIDE]
    display_v = v_values[1:-1:PLOT_STRIDE, ::PLOT_STRIDE]
    display_longitude = longitude[::PLOT_STRIDE]
    arrows = gv.VectorField(
        (
            display_longitude,
            display_latitude,
            np.arctan2(display_v, display_u),
            np.hypot(display_u, display_v),
        ),
        kdims=["longitude", "latitude"],
        vdims=["angle", "magnitude"],
        crs=MAP_CRS,
    ).opts(
        color="#111827",
        line_width=1.0,
        pivot="mid",
        projection=MAP_CRS,
        xlim=(-180, 180),
        ylim=(-90, 90),
    )
    coast = COASTLINE.opts(
        line_color="#374151",
        line_width=0.8,
        projection=MAP_CRS,
    )
    return mesh * arrows * coast


def make_scalar_plot(
    state: dict[str, object], clim: tuple[float, float], title: str
) -> hv.Overlay:
    longitude, latitude, values = plot_scalar_arrays(
        state["field"], float(state["display_scale"])
    )
    mesh = gv.QuadMesh(
        (longitude, latitude, values),
        kdims=["longitude", "latitude"],
        vdims=[state["display_label"]],
        crs=MAP_CRS,
    ).opts(cmap="RdBu_r", colorbar=True, clim=clim, **map_options(title))
    coast = COASTLINE.opts(
        line_color="#374151",
        line_width=0.8,
        projection=MAP_CRS,
    )
    return mesh * coast


def render_panel(
    side: str,
    time_index: int,
    spectral_range: tuple[int, int],
    taper_enabled: bool,
    taper_response: float,
    output_grid: str,
    diagnostic: str,
) -> hv.Overlay:
    lmin, lmax = (int(value) for value in spectral_range)
    taper = float(taper_response) if taper_enabled else None
    original = original_diagnostic(int(time_index), diagnostic)
    processed = processed_diagnostic(
        int(time_index), lmin, lmax, taper, output_grid, diagnostic
    )
    clim = common_color_limits(original, processed)
    state = original if side == "original" else processed
    title = f"{'Original' if side == 'original' else 'Processed'} · {diagnostic}"
    if state["plot_kind"] == "wind":
        return make_wind_plot(state, clim, title)
    return make_scalar_plot(state, clim, title)

## Panel controls and application

The time player updates normally. The expensive spectral range and taper response are bound to `value_throttled`, so transforms run after the user releases a slider. The left and right plots are separate `hv.DynamicMap` objects; their shared state caches keep the same processed field and diagnostic from being recomputed for both panels.

In [ ]:
# ruff: noqa: E501
diagnostic_widget = pn.widgets.Select(
    name="Diagnostic", options=list(DIAGNOSTICS), value="Wind"
)
time_player = pn.widgets.Player(
    name="Time",
    start=0,
    end=len(TIMES) - 1,
    value=0,
    interval=1800,
    loop_policy="loop",
    show_value=False,
)
spectral_widget = pn.widgets.IntRangeSlider(
    name="Spectral degree range",
    start=0,
    end=source_lmax,
    value=(0, DEFAULT_LMAX),
    step=1,
    tooltips=True,
)
taper_enabled_widget = pn.widgets.Checkbox(name="Taper enabled", value=False)
taper_response_widget = pn.widgets.FloatSlider(
    name="Taper endpoint response",
    start=0.01,
    end=1.0,
    step=0.01,
    value=0.1,
    format="0.00",
)
output_grid_widget = pn.widgets.Select(
    name="Output grid", options=list(GRID_MODES), value="ERA5 CC"
)


def set_taper_enabled(event: object) -> None:
    taper_response_widget.disabled = not bool(event.new)


taper_enabled_widget.param.watch(set_taper_enabled, "value")
taper_response_widget.disabled = True


def timestamp_markdown(time_index: int) -> str:
    return f"**ERA5 timestamp:** `{timestamp_label(int(time_index))}`"


def processing_summary(
    time_index: int,
    spectral_range: tuple[int, int],
    taper_enabled: bool,
    taper_response: float,
    output_grid: str,
) -> str:
    lmin, lmax = (int(value) for value in spectral_range)
    taper_text = (
        "hard selection"
        if not taper_enabled
        else f"Sardeshmukh–Hoskins endpoint response = {float(taper_response):g}"
    )
    return (
        f"**Current processing:** `{timestamp_label(int(time_index))}` \n"
        f"`T{lmin}–{lmax}` · {taper_text} · `{output_grid}`"
    )


bindings = {
    "time_index": time_player.param.value,
    "spectral_range": spectral_widget.param.value_throttled,
    "taper_enabled": taper_enabled_widget.param.value,
    "taper_response": taper_response_widget.param.value_throttled,
    "output_grid": output_grid_widget.param.value,
    "diagnostic": diagnostic_widget.param.value,
}
left_plot = hv.DynamicMap(pn.bind(render_panel, side="original", **bindings), kdims=[])
right_plot = hv.DynamicMap(
    pn.bind(render_panel, side="processed", **bindings), kdims=[]
)

timestamp_pane = pn.pane.Markdown(pn.bind(timestamp_markdown, time_player.param.value))
summary_pane = pn.pane.Markdown(
    pn.bind(
        processing_summary,
        time_player.param.value,
        spectral_widget.param.value_throttled,
        taper_enabled_widget.param.value,
        taper_response_widget.param.value_throttled,
        output_grid_widget.param.value,
    )
)

controls = pn.Column(
    pn.pane.Markdown("### Controls"),
    diagnostic_widget,
    time_player,
    timestamp_pane,
    spectral_widget,
    taper_enabled_widget,
    taper_response_widget,
    output_grid_widget,
    pn.pane.Markdown(
        "T0–42 keeps total degrees 0–42. T6–42 removes degrees 0–5. "
        "Taper response is the retained endpoint response."
    ),
    sizing_mode="stretch_width",
)

check_button = pn.widgets.Button(
    name="Run checks for current frame", button_type="primary"
)
check_output = pn.pane.Markdown(
    "Click the button to run current-frame reconstructions and the external ERA5 `vo` reference comparison."
)


def on_check_click(event: object) -> None:
    lmin, lmax = (int(value) for value in spectral_widget.value_throttled)
    taper = (
        float(taper_response_widget.value_throttled)
        if taper_enabled_widget.value
        else None
    )
    try:
        check_output.object = run_consistency_checks(
            int(time_player.value),
            lmin,
            lmax,
            taper,
            output_grid_widget.value,
        )
    except (OSError, RuntimeError, ValueError) as exc:
        check_output.object = f"**Check error:** {exc}"


check_button.on_click(on_check_click)
checks = pn.Card(
    check_button,
    check_output,
    title="Consistency checks",
    collapsed=False,
)

plot_row = pn.Row(
    pn.Column(pn.pane.Markdown("### Original"), left_plot, sizing_mode="stretch_width"),
    pn.Column(
        pn.pane.Markdown("### Processed"), right_plot, sizing_mode="stretch_width"
    ),
    sizing_mode="stretch_width",
)

attribution = pn.pane.Markdown(
    "**Data attribution.** Displayed data are ERA5 from the Copernicus Climate "
    "Change Service / ECMWF, distributed here through the immutable "
    "[PyStormTracker-Data v0.2.0-data](https://github.com/mwyau/PyStormTracker-Data/releases/tag/v0.2.0-data) "
    "release. This example does not imply endorsement by ECMWF or Copernicus. "
    "The optional 0.25° asset is intentionally not used."
)

app = pn.template.FastListTemplate(
    title="spharmgrid — ERA5 850-hPa spherical harmonic explorer"
)
app.sidebar.append(controls)
app.main.append(
    pn.pane.Markdown(
        "Compare the original ERA5 850-hPa wind with a vector spherical harmonic "
        "selection, taper, or fixed-grid regridding on the same timestamp."
    )
)
app.main.append(plot_row)
app.main.append(summary_pane)
app.main.append(checks)
app.main.append(attribution)
app.servable()
app